# Spatial transcriptomics on the GPU: a breast cancer example

An interactive walkthrough of a spatial analysis with rapids-singlecell, on 10x's ATERA breast
cancer dataset (170,057 cells × 18,028 genes).

1. Load the data
2. Quality control and normalization
3. Build the spatial graph (which cells touch)
4. Label cell types from marker genes
5. Find spatial structure: niches, what they're made of, which cell types sit together, ligand-receptor signaling
6. Find genes with spatial patterns
7. Optionally, save the labels and embeddings back into the table

> Needs internet access for the OmniPath lookups in section 5.

Based on rapids-singlecell's [ATERA tutorial](https://rapids-singlecell.scverse.org/en/latest/notebooks/08_atera_demo.html)
by Sara Jimenez, Lukas Heumos and Severin Dicks (scverse). It reads its input from a data asset instead of downloading it.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import time

import numpy as np
import cupy as cp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad
import scanpy as sc
import squidpy as sq 
import rapids_singlecell as rsc
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
import spatialdata as sd
import zarr
from anndata.io import read_elem, write_elem
from annizarr import Repo

import omnipath as op

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 0

# Managed memory lets the GPU spill to host RAM when it fills up, so a large table
# runs slower instead of failing with out_of_memory. Set to False for all-GPU speed.
USE_MANAGED_MEMORY = True
if USE_MANAGED_MEMORY:
    rmm.reinitialize(managed_memory=True, pool_allocator=False)
    cp.cuda.set_allocator(rmm_cupy_allocator)

## 1. Load the data

Set `DATA_PATH` to your data asset. It can be:
- a **SpatialData `.zarr` folder** (unzipped), or
- an **icechunk repo laid out like one** (the table at `tables/<name>`); `BRANCH` picks the branch.

Only the table is used. It needs raw counts in `X`, cell centers in `obsm["spatial"]`, and, for the
QC filter, an `obs` column `nucleus_count` (skipped if missing).

The tutorial used `atera_breast_cancer.zarr.zip` from `exampledata.scverse.org/rapids-singlecell`.
The cell-type markers in section 4 are for breast cancer; other tissues need their own.


In [ ]:
notebook_start = time.time()

In [ ]:
DATA_PATH = "/data/<data asset>/atera_breast_cancer.zarr"
BRANCH = "main"
TABLE = "table"

if Repo.exists(DATA_PATH):
    repo = Repo(DATA_PATH, branch=BRANCH)
    snapshot = repo.log()[0].id
    table = read_elem(repo.open_zarr("r", snapshot_id=snapshot)[f"tables/{TABLE}"])
else:
    sdata = sd.read_zarr(DATA_PATH)
    print(sdata)
    table = sdata.tables[TABLE]
table


Copy the table and move it to the GPU. The raw counts are kept as a layer after filtering (next section),
so only one copy of the matrix is on the GPU while cells and genes are dropped.

In [ ]:
adata = table.copy()
adata.obs["region"] = pd.Categorical(["cells"] * adata.n_obs)  # one library for squidpy's plots
adata.var_names_make_unique()
adata.X = adata.X.astype(np.float32)  # the GPU wants floats

rsc.get.anndata_to_GPU(adata, convert_all=True)
adata


## 2. Quality control and normalization

Compute QC metrics, then drop failed segmentations: cells with almost no counts, or with no nucleus.

In [ ]:
rsc.pp.calculate_qc_metrics(
    adata, 
    expr_type="counts", 
    var_type="genes", 
    log1p=True
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.histplot(adata.obs["total_counts"], bins=100, ax=axes[0])
axes[0].set_title("total_counts (pre-filter)")
sns.histplot(adata.obs["n_genes_by_counts"], bins=100, ax=axes[1])
axes[1].set_title("n_genes_by_counts (pre-filter)")
fig.tight_layout()

In [ ]:
# Drop failed segmentations: empty cells and anucleate fragments.

keep = adata.obs["total_counts"] >= 10
if "nucleus_count" in adata.obs:
    keep &= adata.obs["nucleus_count"] >= 1
adata = adata[keep].copy()
rsc.pp.filter_genes(adata, min_cells=3)
adata.layers["counts"] = adata.X.copy()  # X is still raw counts here; used by highly_variable_genes

Normalize each cell to the same total and take the log; that's the `lognorm` layer used from here on.
Then keep 2,000 highly variable genes and reduce to 50 PCs.

In [ ]:
rsc.pp.normalize_total(adata)
rsc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

In [ ]:
rsc.pp.highly_variable_genes(
    adata, 
    n_top_genes=2000, 
    flavor="seurat_v3", 
    layer="counts"
)

rsc.pp.pca(
    adata, 
    n_comps=50, 
    mask_var="highly_variable", 
    random_state=RANDOM_STATE
)

## 3. Build the spatial graph

Connect each cell to its neighbors by Delaunay triangulation. Triangulation also links cells across
gaps in the tissue, so `percentile=99` drops the longest 1% of edges.

In [ ]:
%%time
rsc.gr.spatial_neighbors_delaunay(adata, percentile=99)

## 4. Label cell types

There's no reference atlas here, so cells are labelled by **marker-gene scores** (`rsc.dcg.ulm`).
The scores are averaged within each leiden cluster, and each cluster gets the best-scoring type,
which is far more robust than labelling cells one by one.

The tumor markers come from Janesick et al. 2026 (AACR poster on this dataset).

In [ ]:
CELL_TYPE_MARKERS: dict[str, list[str]] = {
    # -- tumour epithelium (Janesick et al. AACR 2026 poster) --
    "Invasive_Tumor": ["CCND1", "MYEOV", "TPCN2", "LTO1", "SMIM38", "FGF19", "FGF4", "FADD", "CTTN"],  # 11q13, Fig. 9C
    "Invasive_Tumor_G1S": ["PCNA", "MCM2", "MCM5", "TYMS", "FEN1", "GINS2"],              # S-phase, Fig. 8
    "Invasive_Tumor_Mitotic": ["C21orf58", "ESPL1", "MELK", "NCAPG2", "TOP2A", "CENPF"],  # Fig. 8 + G2/M
    "DCIS_Basal_like": ["KIT", "PROM1", "GABRP", "SYNM", "CCL28"],                        # Fig. 7A
    "DCIS_Luminal_like": ["ANKRD30A", "ESR1", "NIBAN1", "BMPR1B", "SEC23B", "SLC24A3", "BARX2", "TSKU"],  # Fig. 7B
    "Apocrine": ["PIP", "AR", "HPGD", "ACSM1"],                                           # PIP per Fig. 6
    "Myoepithelial": ["TP63", "OXTR", "KRT14", "KRT17", "KRT5", "ACTA2", "MYLK"],         # epithelial + contractile
    # -- stroma / vasculature --
    "CAF_Neoplastic": ["PDGFRA", "DCN", "LUM", "COL1A2", "CFD", "C7"],          # pan-CAF + non-activated
    "CAF_Invasive": ["COL11A1", "COL10A1", "MMP11", "INHBA", "THBS2", "CTHRC1"], # invasion-associated stroma
    "CXCL14_pos": ["CXCL14"],                                                   # only marker given in Fig. 6
    "Pericyte": ["RGS5", "NOTCH3", "KCNJ8", "HIGD1B", "ABCC9"],                 # PDGFRB dropped (CAFs)
    "Endothelial": ["PECAM1", "VWF", "CDH5", "CLDN5", "EMCN", "PLVAP"],
    # -- immune --
    "T_Cell": ["CD3D", "CD3E", "CD3G", "CD2", "CD247", "CD6"],                  # CD4 / IL7R dropped
    "B_Cell": ["MS4A1", "CD19", "PAX5", "BANK1", "BLK", "FCRL1"],               # Fig. 4B genes; CD79A/B dropped
    "Plasma_Cell": ["MZB1", "JCHAIN", "TNFRSF17", "DERL3"],                     # non-Ig -> less spill-over
    "Macrophage": ["CD68", "CD163", "C1QA", "C1QB", "C1QC", "MS4A7"],
    "Dendritic_Cell": ["FLT3", "CLEC9A", "XCR1", "CD1C", "CLEC10A", "LAMP3"],
    # -- not labelled in Fig. 6: candidates for its 2 small unlabelled clusters --
    "Lymphatic_Endothelial": ["PROX1", "CCL21", "MMRN1", "FLT4", "LYVE1"],     # PDPN dropped
    "Mast_Cell": ["TPSAB1", "TPSB2", "CPA3", "MS4A2", "HDC"],                  # KIT left to DCIS_Basal_like
}

celltype_net = (
    pd.Series(CELL_TYPE_MARKERS).explode().rename_axis("source").reset_index(name="target")
)
celltype_net.groupby("source").size()

In [ ]:
adata.X = adata.layers["lognorm"]

rsc.dcg.ulm(adata, celltype_net, layer="lognorm", tmin=2)
adata.obsm["score_celltype"] = adata.obsm.pop("score_ulm")
adata.obsm.pop("padj_ulm", None)

score_ct = adata.obsm["score_celltype"]

In [ ]:
rsc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca", random_state=RANDOM_STATE)
rsc.tl.leiden(adata, resolution=1.0, key_added="leiden_expr", random_state=RANDOM_STATE)
rsc.tl.umap(adata, random_state=RANDOM_STATE)

In [ ]:
# Robust label: mean decoupler score per expression cluster
cluster_mean_score = score_ct.groupby(adata.obs["leiden_expr"].values).mean()
cluster_to_celltype = cluster_mean_score.idxmax(axis=1)

adata.obs["cell_type"] = (
    adata.obs["leiden_expr"].map(cluster_to_celltype).astype("category")
)

adata.obs["cell_type"].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sc.pl.umap(adata, color="leiden_expr", ax=axes[0], show=False, legend_loc="on data", title="Leiden (expression)")
sc.pl.umap(adata, color="cell_type", ax=axes[1], show=False, title="Cell type (cluster-level argmax)")

In [ ]:
# Categorical spatial layout via squidpy
sq.pl.spatial_scatter(
    adata, library_id="region", color="cell_type", shape=None, img=False,
    palette="tab20", size=1.5, figsize=(7, 6),
)

## 5. Spatial structure

### 5.1 Niches

`rsc.gr.calculate_niche_cellcharter` (CellCharter) groups cells by their own expression plus that of
their neighbors up to 3 steps away, then fits a mixture model with `n_components` niches.

- **It works on the 50 PCs, not all genes.** On all 18,028 genes the GPU runs out of resources;
  the original CellCharter paper also uses PCs.
- **9 niches** separates tumor, stromal and immune areas while staying readable, close to the ~10
  reported for this dataset.

In [ ]:
%%time 

niche_ad = ad.AnnData(
    X=rsc.get.X_to_CPU(adata.obsm["X_pca"]),
    obs=pd.DataFrame(index=adata.obs_names),
)

niche_ad.obsp["spatial_connectivities"] = adata.obsp["spatial_connectivities"]

rsc.gr.calculate_niche_cellcharter(niche_ad, n_components=9, random_state=RANDOM_STATE)

adata.obs["cellcharter_niche"] = niche_ad.obs["cellcharter_niche"].astype("category").values

adata.obs["cellcharter_niche"].value_counts()

In [ ]:
sq.pl.spatial_scatter(
    adata, 
    library_id='region',
    color="cellcharter_niche", 
    shape=None, 
    img=False,
    size=1 
)

### 5.2 What each niche is made of

- **Composition:** the mix of cell types in each niche.
- **Pathway activity:** the mean PROGENy score per niche for 14 signaling pathways, downloaded from OmniPath.

In [ ]:
composition = (
    pd.crosstab(adata.obs["cellcharter_niche"], adata.obs["cell_type"], normalize="index") * 100
)

fig, ax = plt.subplots(figsize=(10, 5))
composition.plot(kind="bar", stacked=True, ax=ax)
ax.set_ylabel("% of cells in niche")
ax.set_title(f"Cell-type composition per spatial domain")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
fig.tight_layout()

composition.round(1)

In [ ]:
# PROGENy pathway footprints, retrieved from OmniPath. Weights are signed, which is
# what ulm expects; the curated lists this replaces were effectively unweighted.

N_PROGENY_GENES = 500

progeny_net = (
    op.requests.Annotations.get(resources="PROGENy")
    .pivot(index=["record_id", "genesymbol"], columns="label", values="value")
    .reset_index()
    .rename(columns={"pathway": "source", "genesymbol": "target"})
    .astype({"weight": float, "p_value": float})
    .sort_values("p_value")
    .drop_duplicates(subset=["source", "target"])
    .groupby("source")
    .head(N_PROGENY_GENES)[["source", "target", "weight"]]
    .reset_index(drop=True)
)

rsc.dcg.ulm(adata, progeny_net, layer="lognorm")

adata.obsm["score_pathway"] = adata.obsm.pop("score_ulm")

adata.obsm.pop("padj_ulm", None)

score_pw = adata.obsm["score_pathway"]
niche_pathway = score_pw.groupby(adata.obs["cellcharter_niche"].values).mean()
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(niche_pathway, cmap="vlag", center=0, annot=True, fmt=".1f", ax=ax)
ax.set_title("Mean PROGENy pathway activity per spatial domain")
fig.tight_layout()

### 5.3 Which cell types sit together

Niches describe neighborhoods; these describe pairs of cell types.

- **Co-occurrence** (`rsc.gr.co_occurrence`): how likely another cell type is at each distance from a given one.
- **Neighborhood enrichment** (`rsc.gr.nhood_enrichment`): which pairs are neighbors more or less often than chance.
- **Ligand-receptor signaling** (`rsc.gr.ligrec`): ligand-receptor pairs enriched between cell types,
  from OmniPath's ~11k interactions. Every pair is tested, which takes seconds on the GPU, and only the
  most consistently significant ones are plotted.

In [ ]:
%%time

rsc.gr.co_occurrence(
    adata,
    cluster_key="cell_type",
    interval=np.linspace(0, 250, 26),
    multi_gpu=False,
)
sq.pl.co_occurrence(
    adata,
    cluster_key="cell_type",
    clusters="Invasive_Tumor",
    figsize=(7, 5),
)

In [ ]:
rsc.gr.nhood_enrichment(adata, cluster_key="cell_type", n_perms=500, seed=RANDOM_STATE)
sq.pl.nhood_enrichment(adata, cluster_key="cell_type", figsize=(8, 7))

In [ ]:
interactions = rsc.squidpy_gpu._ligrec._get_interactions()

In [ ]:
%%time
lr_genes = (
    interactions[["source", "target"]]
    .stack()
    .astype(str)
    .str.split("_")
    .explode()
    .unique()
)
lr_genes = adata.var_names[adata.var_names.isin(lr_genes)]

adata_lr = adata[:, lr_genes].copy()
adata_lr.X = adata_lr.layers["lognorm"]
adata_lr.obs["cell_type"] = adata_lr.obs["cell_type"].cat.remove_unused_categories()
rsc.get.anndata_to_CPU(adata_lr, convert_all=True)  # ligrec's setup is host-side pandas; it moves the matrix to GPU itself

cp.random.seed(RANDOM_STATE)  # rsc.gr.ligrec takes no seed argument
ligrec_res = rsc.gr.ligrec(
    adata_lr,
    n_perms=1000,
    interactions=interactions,
    cluster_key="cell_type",
    copy=True,
    use_raw=False,
    corr_method="fdr_bh",
)
del adata_lr

In [ ]:
lr_clusters = [
    "Invasive_Tumor", "DCIS_Basal_like", "T_Cell",
    "Macrophage", "CAF_Neoplastic", "Endothelial",
]

ALPHA, TOP_N, MIN_SIG = 0.01, 10, 5

pv, mu = ligrec_res["pvalues"], ligrec_res["means"]

# Rank only over the cluster pairs that will actually be drawn.
cols = [c for c in pv.columns if c[0] in lr_clusters and c[1] in lr_clusters]

# Densify first: boolean reductions on sparse frames are not reliable across pandas versions.
pv_d, mu_d = pv[cols].sparse.to_dense(), mu[cols].sparse.to_dense()
sig = pv_d <= ALPHA
rank = pd.DataFrame({
    "n_sig": sig.sum(axis=1),
    "max_mean": mu_d.where(sig).max(axis=1),
})
# Keep interactions significant in at least MIN_SIG cluster pairs, then rank by
# their strongest significant mean expression.
top = (
    rank[rank["n_sig"] >= MIN_SIG]
    .sort_values("max_mean", ascending=False)
    .head(TOP_N)
    .index
)
ligrec_top = {k: df.loc[top] for k, df in ligrec_res.items()}

sq.pl.ligrec(
    ligrec_top,
    source_groups=lr_clusters,
    target_groups=lr_clusters,
    alpha=ALPHA,
    pvalue_threshold=ALPHA,
    title="Top ligand-receptor interactions across cell-type pairs",
    figsize=(0.7 * len(lr_clusters) + 4, 4),
)

## 6. Genes with spatial patterns

Moran's I scores how strongly a gene's expression clusters in space (`rsc.gr.spatial_autocorr`).

In [ ]:
%%time
moran_df = rsc.gr.spatial_autocorr(adata, mode="moran", layer="lognorm", copy=True)
moran_df = moran_df.sort_values("I", ascending=False)
moran_df.head(10)

In [ ]:
%%time
top_svgs = moran_df.index[:6].tolist()

# squidpy plotting is CPU/matplotlib-based and can't read CuPy-backed
# layers directly, so pull this AnnData back to host once before plotting 

rsc.get.anndata_to_CPU(adata, convert_all=True)

titles = {g: f"{g}  (Moran's I={moran_df.loc[g, 'I']:.2f})" for g in top_svgs}
axes = sq.pl.spatial_scatter(
    adata, 
    library_id='region',
    color=top_svgs, 
    layer="lognorm", 
    shape=None, 
    img=False,
    cmap="viridis", 
    size=1.5, 
    ncols=3, 
    figsize=(15, 9), 
    return_ax=True,
)
for ax, gene in zip(np.atleast_1d(axes), top_svgs):
    ax.set_title(titles[gene])

`CCND1` is the most spatially patterned gene (Moran's I = 0.73). It's highest in the large invasive
tumor nests and low in the in-situ ducts and stroma, matching the CCND1 gain seen only in invasive
cells by Janesick et al. 2026.

## 7. Save the results to the table (optional)

This writes the cell types, clusters, niches and embeddings into the store's table, under their own
names (`spatial_*`, `X_*_spatial`) so they don't replace the `run` pipeline's results. The images,
shapes and points are untouched. Cells removed by QC get empty values.

- **icechunk:** one commit on `BRANCH`. Earlier versions stay in the history. If the branch moved since
  section 1, nothing is written; rerun on the new tip.
- **SpatialData `.zarr`:** in place.

A data asset under `/data` is read-only. To save, use a writable copy, e.g. in `/scratch` (for icechunk,
`Repo(DATA_PATH, branch=BRANCH, origin="s3://...")` writes to the repo's writable origin instead).
Set `SAVE = True` to write.


In [ ]:
SAVE = False
SAVE_OBS = {"cell_type": "spatial_cell_type", "leiden_expr": "spatial_leiden", "cellcharter_niche": "spatial_niche"}
SAVE_OBSM = {"X_pca": "X_pca_spatial", "X_umap": "X_umap_spatial"}


def save_to_table(group):
    index = read_elem(group["obs"]).index
    rows = index.get_indexer(adata.obs_names)
    assert index.is_unique and (rows >= 0).all(), "the table's cells don't match the ones analyzed"
    obs = group["obs"]
    for column, name in SAVE_OBS.items():
        write_elem(obs, name, pd.Categorical(adata.obs[column].reindex(index)))
    obs.attrs["column-order"] = [*(c for c in obs.attrs["column-order"] if c not in SAVE_OBS.values()),
                                 *SAVE_OBS.values()]
    for key, name in SAVE_OBSM.items():
        full = np.full((len(index), adata.obsm[key].shape[1]), np.nan, dtype=np.float32)
        full[rows] = adata.obsm[key]
        write_elem(group["obsm"], name, full)


if SAVE and Repo.exists(DATA_PATH):
    assert repo.log()[0].id == snapshot, f"branch {BRANCH} moved since section 1; rerun on the new tip"
    save_to_table(repo.open_zarr("w")[f"tables/{TABLE}"])
    print(repo.commit("Spatial notebook: cell types, niches, UMAP"))
elif SAVE:
    save_to_table(zarr.open_group(DATA_PATH, mode="r+", use_consolidated=False)[f"tables/{TABLE}"])
    zarr.consolidate_metadata(DATA_PATH)


In [ ]:
print("Total Processing time: %s" % (time.time() - notebook_start))